In [30]:
#use to quit Engines
aspen.Close()
engine.quit()

In [24]:
#importing all the neccessary libraries
import win32com.client as win32
#import the matlab engine
import matlab.engine
import os
import pandas as pd
from DAC_helpers import obtain_ergun_pressure_inlet

In [25]:
#Start Aspen controlling session

#Start Aspen
aspen = win32.gencache.EnsureDispatch("Apwn.Document")
print("Aspen Engine started")

#start matlab engine and continue sesssion
engine = matlab.engine.start_matlab()
print("MATLAB Engine started")

#add path to allow for easy finding of matlab files
engine.addpath(r"C:\Users\User\Documents\MATLAB", nargout=0)


aspen_sim_path = r"C:\Users\User\Documents\AspenTech\Aspen Plus V14.0\DAC_simulation_V2.bkp"
#Load simulation file
aspen.InitFromFile2(aspen_sim_path)
print(f"Aspen Simulation file in path: {aspen_sim_path} loaded and Aspen plus application opened.")

#Make aspen visible
aspen.Visible = True

#Suppress dialog
aspen.SuppressDialogs = True

Aspen Engine started
MATLAB Engine started
Aspen Simulation file in path: C:\Users\User\Documents\AspenTech\Aspen Plus V14.0\DAC_simulation_V2.bkp loaded and Aspen plus application opened.


In [26]:
# Read in data frame of LHS values
df_simulation_inputs = pd.read_csv("DAC_optimal_inputs.csv")
print("Latin Hypercube Sampling Values read into DataFrame")

# Check if the path of the file with the simulation result exists
print("Checking for a DAC simulation results csv")
if os.path.exists("DAC_optimal_simulation_results_testing.csv"):
    df_simulation_results = pd.read_csv("DAC_optimal_simulation_results_testing.csv")
    print("Yes, DAC optimal simulation results csv exists")

    # store index of the simulation results data frame in a list
    completed_sim_results_indicies = df_simulation_results["sim_index"].to_list()


#If the path of the file with the simulation results does not exist, create it
else:
    #Create an empty data frame that will hold the simulation results
    df_simulation_results = pd.DataFrame()

    #create an empty list that will contain the indicies of completed simulation
    completed_sim_results_indicies = []

print(f" Simulation results exist for {len(completed_sim_results_indicies)} runs")

#Start for loop that runs through the rows in the df_simulation_inputs LHS values
for (index, row) in df_simulation_inputs.iterrows():
    
    #Check if the current index is in completed simulation indexes, skip and move to the next index if satisfied
    if index in completed_sim_results_indicies:
        #skip and move to the next index
        continue
    
    #Now on index that does not yet exist
    print(f"Preparing simulation for index {index}")
    
    #Re intitialize the simulation before inputing values and running simulation
    # Do this at the start of every loop
    aspen.Engine.Reinit()
    print("Re initialized Aspen simulation to clear existing iterations")

    print("Now writing values into the desired fields in Aspen plus")
    #Writing LHS variables into desired Aspen plus fields
    
    #Writing in the feed flow in kmol/hr of CaCO3
    caco3_feed_flow_rate = row["caco3_flow_rate [kmol/hr]"]
    aspen.Tree.FindNode(r"\Data\Streams\CACO3\Input\TOTFLOW\CISOLID").Value = caco3_feed_flow_rate

    #Writing in and setting calciner temperature
    calciner_temperature = row["calciner_temp [Deg C]"]
    aspen.Tree.FindNode(r"\Data\Blocks\CALCINER\Input\TEMP").Value = calciner_temperature
    
    # Writing in the feed flow of Water, water that flows into the Slaker.
    #This equals the flow rate of CaO (Which is the flow rate of CaCO3) + 0.001
    # flow rate of water into the slaker
    slaker_water_in = caco3_feed_flow_rate + 0.001
    #print(slaker_water_in)
    #writing to aspen
    aspen.Tree.FindNode(r"\Data\Streams\WATER\Input\TOTFLOW\MIXED").Value = slaker_water_in

    #feed flow of calcium hyroxide that flows out from slaker into the mixer
    caOH2_flow = caco3_feed_flow_rate

    #Calculate the flow rate of water into mixer to achieve the desired hydration moisture fraction
    #first read in the desired moisture fraction
    pellet_water_fraction = row["pellet_moisture_frac"]
    #flow rate of water into mixer
    mixer_water_in = ((pellet_water_fraction / 18.02 * (1 - pellet_water_fraction)) * caOH2_flow * 74.09) - 0.001
    #Now write the mixer water flow rate into the proper aspen plus field
    aspen.Tree.FindNode(r"\Data\Streams\WATER2\Input\TOTFLOW\MIXED").Value = mixer_water_in

    #Write in the flow rate of air into the desired blower field
    air_flow = row["air_mol_flow_rate [kmol/hr]"]
    # air_flow entering the blower
    aspen.Tree.FindNode(r"\Data\Streams\AIR\Input\TOTFLOW\MIXED").Value

    #Write in the vapor fraction of humidifier
    RH = row["RH_air"]
    # RH in humidifier, calculate the mole flow rate of water vapor that gives the desired humidity fraction
    vap_mol = ((RH / 18.02 * (1 - RH)) * air_flow* 28.97)


    # read in the flow rate of water vapor
    #use relative humidity to determine the moles of water vapor that gives the humidity
    aspen.Tree.FindNode(r"\Data\Streams\H2O-VAP\Input\TOTFLOW\MIXED").Value = vap_mol
    

    #Writing in the DAC temperature
    #Read in temperature of DAC unit from lhs
    DAC_temp = row["DAC_ambient_temp [Deg C]"]
    #write the value in aspen DAC unit
    #DAC_temp in DAC unit
    aspen.Tree.FindNode("\Data\Blocks\DAC-UNIT\Input\TEMP").Value = DAC_temp
    



    #RUNNING THE SIMULATION AFTER WRITING IN "MOST" OF THE DESIRED VARIABLES INTO ASPEN PLUS FIELD
    #RUNNING THE SIMULATION. THIS STAGE IS CRUCIAL FOR THE HANDSHAKE BETWEEN MATLAB AND ASPEN PLUS
    #RUNNING THE SIMULATION. SOME VALUES WILL BE EXTRACTED FROM SOME ASPEN RESULTS FIELDS AND USED IN MATLAB SCM AND ERGUN PRESSURE DROP COMP.
    #RUNNING THE SIMULATION. PRODEEDS

    try:
        #This runs the aspen plus simulation
        aspen.Engine.Run2()
        print("Running Aspen plus simulation, trial 1/2")

        #Check the status of the simulation before grabbing results to see if the simulation converged. 0 is success
        convergence_status = aspen.Tree.FindNode(r"\Data\Results Summary\Run-Status\Output\PER_ERROR").Value
        if (convergence_status == 0) or (convergence_status==1):
            #Grab the reults immediately
            #Grab the volumetric flow rates of the pellets read in s litre/min. convert to m3/hr
            pellet_vol_flow = aspen.Tree.FindNode(r"\Data\Streams\PELLET\Output\RES_VOLFLOW").Value / 60000
            
            #Grab the results of the molecular weight of air 
            MW_air = aspen.Tree.FindNode(r"\Data\Streams\WET-AIR\Output\MW").Value
            
            #Grab the results of the volumetric flow rate of air leaving the humidifer and entering the DAC unit in l/mins. convert to m3/hr
            air_vol_flow_rate = aspen.Tree.FindNode(r"\Data\Streams\WET-AIR\Output\RES_VOLFLOW").Value / 60000

            #Read in the mass flow rate of wet air entering the DAC unit
            air_kg_per_hr = aspen.Tree.FindNode(r"\Data\Streams\WET-AIR\Output\RES_MASSFLOW").Value
            
            #Read in the mole fraction of co2 in air leaving humidifier and entering DAC unit
            y_co2 = aspen.Tree.FindNode(r"\Data\Streams\WET-AIR\Output\MOLEFRAC\MIXED\CO2").Value

            #Read in temperature of DAC unit from lhs
            DAC_temp = row["DAC_ambient_temp [Deg C]"]

            #Read in length of pipe from lhs
            height_DAC = row["DAC_column_height [m]"]

            #Read in the value of the voidage
            e_void = row["e_voidage"]

            #Read in the diameters of pellet from lhs
            pellet_diameter = pellet_diameter = row["pellet_diameter [mm]"]

            #Mole of CO2 into the DAC unit from the humidifier
            nco2 = aspen.Tree.FindNode(r"\Data\Streams\WET-AIR\Output\MOLEFLOW\MIXED\CO2").Value
            

            #Compute the Ergun pressue Inlet
            P_in_DAC =  obtain_ergun_pressure_inlet(L=height_DAC, e_void=e_void, dp=pellet_diameter, DAC_ambient_temp=DAC_temp, kg_per_hr=air_kg_per_hr, MW_air=MW_air)

            L= height_DAC
            npell= aspen.Tree.FindNode(r"\Data\Streams\PELLET\Output\MOLEFLMX\CISOLID").Value
            Vpell = pellet_vol_flow
            #Average pressure in the DAC unit. To be used MATLAB SCM
            P_ave = (2/3) * (P_in_DAC**2 + 1**2 + P_in_DAC * 1) / (P_in_DAC + 1)

            
            #MATLAB AND ASPEN PLUS HANDSHAKE. PYTHON SENDS VALUES TO MATLAB, MATLAB COMPUTES CONVERSION
            #MATLAB THEN SENDS CONVERSION TO PYTHON
            #PYTHON THEN WRITES THE CONVERSION INTO ASPEN AND RUNS THE SIMULATION THE SECOND TIME
            #RUN MATLAB ENGINE
            
            x_conversion, x_conv_pell = engine.DAC_func(matlab.double(pellet_diameter*0.001), matlab.double(y_co2), matlab.double(P_ave*101325), matlab.double(DAC_temp+273.15), matlab.double(RH), matlab.double(e_void), matlab.double(Vpell), matlab.double(L), matlab.double(npell), matlab.double(nco2), nargout=2)


            #print("MATLAB SIMULATION COMPLETED")

            #Write conversion into Aspen field
            aspen.Tree.FindNode(r"\Data\Blocks\DAC-UNIT\Input\CONV\1").Value = x_conversion

            #Set the pressure of pump and air blower equal to P_in_DAC
            # discharge presure of Air Blower unit
            aspen.Tree.FindNode(r"\Data\Blocks\AIR-BLOW\Input\PRES").Value = P_in_DAC
            
            # setting the inlet presusure of steam unit
            aspen.Tree.FindNode(r"\Data\Streams\H2O-VAP\Input\PRES\MIXED").Value = P_in_DAC

            ##RUN THE SECOND TRIAL OF SIMULATION
            ##RUN THE SECOND TRIAL OF SIMULATION
            ##RUN THE SECOND TRIAL OF SIMULATION
            ##RUN THE SECOND TRIAL OF SIMULATION
            ##RUN THE SECOND TRIAL OF SIMULATION
            ##RUN THE SECOND TRIAL OF SIMULATION
            ##RUN THE SECOND TRIAL OF SIMULATION
            
            aspen.Engine.Run2()
            print("Running Aspen plus simulation, trial 2/2")
            
            #Read the mass flow rate of co2 leaving the DAC unit
            #mass flow of co2 leaving the DAC unit
            mass_flow_co2_out = aspen.Tree.FindNode(r"\Data\Streams\CO2-LEAN\Output\MASSFLOW\MIXED\CO2").Value

            #Read in the mass flow rate of co2 into the DAC unit
            mass_flow_co2_in = aspen.Tree.FindNode(r"\Data\Streams\WET-AIR\Output\MASSFLOW\MIXED\CO2").Value

            #Read in the pellet flow rate
            calcium_hydroxide_pellet_kmol_per_hr = aspen.Tree.FindNode(r"\Data\Streams\PELLET\Output\RES_MOLEFLOW").Value

            #Read in the heat duty or work for each block
            E_duty_calciner = aspen.Tree.FindNode(r"\Data\Blocks\CALCINER\Output\QNET").Value

            #Slaker heat management cost is negligible
            #E_duty_slaker = aspen.Tree.FindNode(r"\Data\Blocks\SLAKER\Output\QNET").Value

            #Humidifier has negligible heat duty
            #E_duty_humidifier = aspen.Tree.FindNode(r"\Data\Blocks\HUMIDIFY\Output\QNET").Value

            #E_duty_DAC_unit = aspen.Tree.FindNode(r"\Data\Blocks\DAC-UNIT\Output\QNET").Value

            E_brake_air_blower = aspen.Tree.FindNode(r"\Data\Blocks\AIR-BLOW\Output\BRAKE_POWER").Value

            #K is a heat reintegration constant, obtained observation the heat duty required to raise a kmol/hr of caco3 to 450C.
            #Helps to estimate the heat from DAC unit that can be reintegrated back to the calciner

            k = 3.4746e-4


            #calculate total heat duty
            E_duty = abs(E_duty_calciner) - (caco3_feed_flow_rate / k)

            #calculate the total brake power
            E_brake_power = E_brake_air_blower


            #include results in current dictionary of results, first sure make to include the rows in the lhs
            curr_results = row.to_dict()

            curr_results["Simulation_status"] = "Success"
            curr_results["Failure_reason"] = None
            
            curr_results["Heat_duty[cal/sec]"] = E_duty
            curr_results["Brake_power[kW]"] = E_brake_power

            curr_results["calcium_hydroxide_pellet[kmol_per_hr]"] = calcium_hydroxide_pellet_kmol_per_hr

            curr_results["SCM_conversion_fraction"] = x_conversion
            curr_results["Ca(OH)2_conversion"] = x_conv_pell

            curr_results["mass_flow_co2_in[kg/hr]"] = mass_flow_co2_in

            curr_results["mass_flow_co2_out[kg/hr]"] = mass_flow_co2_out

            curr_results["DAC_inlet_press[atm]"] = P_in_DAC

            print(f"completed simulation run for simulation {index}")


            #convert to data frame
            df_current =  pd.DataFrame([curr_results])

            header_needed = not os.path.exists("DAC_optimal_simulation_results_testing.csv")

            #Append the data frame to
            df_current.to_csv("DAC_optimal_simulation_results_testing.csv", mode='a', index=False, header=header_needed)
            print("Added result of current simulation run to the DAC Simulation csv file")
            
    
        else:
            print("Aspen plus simulation trial 1/2 did NOT converge successfully")
            #include results in current dictionary of results, first sure make to include the rows in the lhs
            curr_results = row.to_dict()

            curr_results["Simulation_status"] = "Failure"
            curr_results["Failure_reason"] = "Simulation did not converge"
            curr_results["Ca(OH)2_conversion"] = None
            
            curr_results["Heat_duty[cal/sec]"] = None
            
            curr_results["Brake_power[kW]"] = None
            
            curr_results["calcium_hydroxide_pellet[kmol_per_hr]"] = None

            curr_results["SCM_conversion_fraction"] = None

            curr_results["mass_flow_co2_in[kg/hr]"] = None

            curr_results["mass_flow_co2_out[kg/hr]"] = None

            curr_results["DAC_inlet_press[atm]"] = None


            #convert to data frame
            df_current =  pd.DataFrame([curr_results])

            header_needed = not os.path.exists("DAC_optimal_simulation_results_testing.csv")

            #Append the data frame to
            df_current.to_csv("DAC_optimal_simulation_results_testing.csv", mode='a', index=False, header=header_needed)

            
            

        
    except Exception as e:
        
        print("Aspen plus simulation trial 1/2 did NOT converge successfully")
        #include results in current dictionary of results, first sure make to include the rows in the lhs
        curr_results = row.to_dict()

        curr_results["Simulation_status"] = "Failure"
        curr_results["Failure_reason"] = str(e)
            
        curr_results["Heat_duty[cal/sec]"] = None
        curr_results["Brake_power[kW]"] = None

        curr_results["SCM_conversion_fraction"] = None
        
        curr_results["calcium_hydroxide_pellet[kmol_per_hr]"] = None
        
        curr_results["Ca(OH)2_conversion"] = None

        curr_results["mass_flow_co2_in[kg/hr]"] = None

        curr_results["mass_flow_co2_out[kg/hr]"] = None

        curr_results["DAC_inlet_press[atm]"] = None


        #convert to data frame
        df_current =  pd.DataFrame([curr_results])

        header_needed = not os.path.exists("DAC_optimal_simulation_results_testing.csv")

        #Append the data frame to
        df_current.to_csv("DAC_optimal_simulation_results_testing.csv", mode='a', index=False, header=header_needed) #this line
        
        print(e)
print("all Simulations completed successfully")
aspen.Close()
engine.quit()

Latin Hypercube Sampling Values read into DataFrame
Checking for a DAC simulation results csv
 Simulation results exist for 0 runs
Preparing simulation for index 0
Re initialized Aspen simulation to clear existing iterations
Now writing values into the desired fields in Aspen plus
Running Aspen plus simulation, trial 1/2
Running Aspen plus simulation, trial 2/2
completed simulation run for simulation 0
Added result of current simulation run to the DAC Simulation csv file
Preparing simulation for index 1
Re initialized Aspen simulation to clear existing iterations
Now writing values into the desired fields in Aspen plus
Running Aspen plus simulation, trial 1/2
Running Aspen plus simulation, trial 2/2
completed simulation run for simulation 1
Added result of current simulation run to the DAC Simulation csv file
Preparing simulation for index 2
Re initialized Aspen simulation to clear existing iterations
Now writing values into the desired fields in Aspen plus
Running Aspen plus simulatio

In [27]:
pd.read_csv("DAC_optimal_inputs.csv")

,caco3_flow_rate [kmol/hr],calciner_temp [Deg C],pellet_moisture_frac,air_mol_flow_rate [kmol/hr],RH_air,DAC_column_height [m],pellet_diameter [mm],DAC_ambient_temp [Deg C],e_voidage,capture_efficiency_%,SECC_$
0,97.863788,919.904840,0.064531,7118.821754,0.870483,9.997187,1.049191,42.714769,0.353125,65.621880,10.950542
1,97.924051,919.904840,0.064555,7118.821754,0.870483,9.997187,1.049191,42.716989,0.353125,65.621880,10.950542
2,97.832777,919.696195,0.064504,7117.219329,0.872091,9.999902,1.059883,43.707146,0.353171,65.603966,7.172618
3,97.837663,919.099393,0.064516,7114.636077,0.870726,9.995373,1.059244,43.733019,0.353064,65.585068,5.621132
4,97.898044,919.696195,0.064609,7045.451161,0.870730,9.995326,1.060114,43.722263,0.353059,65.578468,3.874981
5,97.621017,919.402894,0.065140,7041.773748,0.872018,9.996710,1.045518,43.740941,0.353145,65.559578,2.323490
6,97.806708,919.733927,0.064599,7117.490152,0.870721,9.999612,1.057005,43.717158,0.353101,65.603966,7.172618
7,97.667482,919.694791,0.064772,7042.829655,0.898222,9.966109,1.047911,43.729616,0.353064,65.494453,1.046576
8,97.904062,919.841700,0.064597,7044.707805,0.870339,9.999844,1.060147,43.717158,0.353101,65.578468,3.874981
9,97.837736,919.099393,0.064516,7105.147999,0.870726,9.995367,1.059244,43.733019,0.353064,65.585068,5.621132


In [28]:
pd.read_csv("DAC_optimal_simulation_results_testing.csv")

,caco3_flow_rate [kmol/hr],calciner_temp [Deg C],pellet_moisture_frac,air_mol_flow_rate [kmol/hr],RH_air,DAC_column_height [m],pellet_diameter [mm],DAC_ambient_temp [Deg C],e_voidage,capture_efficiency_%,...,Simulation_status,Failure_reason,Heat_duty[cal/sec],Brake_power[kW],calcium_hydroxide_pellet[kmol_per_hr],SCM_conversion_fraction,Ca(OH)2_conversion,mass_flow_co2_in[kg/hr],mass_flow_co2_out[kg/hr],DAC_inlet_press[atm]
0,97.863788,919.904840,0.064531,7118.821754,0.870483,9.997187,1.049191,42.714769,0.353125,65.621880,...,Success,NaN,1.457212e+06,17913.0026,122.153747,0.976727,0.029466,129.933354,3.023947,4.742487
1,97.924051,919.904840,0.064555,7118.821754,0.870483,9.997187,1.049191,42.716989,0.353125,65.621880,...,Success,NaN,1.458110e+06,17913.0905,122.237356,0.976974,0.029455,129.933354,2.991797,4.742516
2,97.832777,919.696195,0.064504,7117.219329,0.872091,9.999902,1.059883,43.707146,0.353171,65.603966,...,Success,NaN,1.456600e+06,17828.4877,122.105594,0.964936,0.029120,129.933354,4.555988,4.714537
3,97.837663,919.099393,0.064516,7114.636077,0.870726,9.995373,1.059244,43.733019,0.353064,65.585068,...,Success,NaN,1.456242e+06,17851.8849,122.115774,0.963974,0.029089,129.933354,4.681037,4.722263
4,97.898044,919.696195,0.064609,7045.451161,0.870730,9.995326,1.060114,43.722263,0.353059,65.578468,...,Success,NaN,1.457572e+06,17830.2229,122.223889,0.963410,0.029054,129.933354,4.754213,4.715110
5,97.621017,919.402894,0.065140,7041.773748,0.872018,9.996710,1.045518,43.740941,0.353145,65.559578,...,Success,NaN,1.453236e+06,17942.9735,122.063365,0.981104,0.029672,129.933354,2.455183,4.752427
6,97.806708,919.733927,0.064599,7117.490152,0.870721,9.999612,1.057005,43.717158,0.353101,65.603966,...,Success,NaN,1.456239e+06,17872.1426,122.106236,0.966676,0.029180,129.933354,4.329906,4.728959
7,97.667482,919.694791,0.064772,7042.829655,0.898222,9.966109,1.047911,43.729616,0.353064,65.494453,...,Failure,"(-2147352567, 'Exception occurred.', (14, 'VAS...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,97.904062,919.841700,0.064597,7044.707805,0.870339,9.999844,1.060147,43.717158,0.353101,65.578468,...,Success,NaN,1.457766e+06,17833.3703,122.227054,0.963125,0.029044,129.933354,4.791345,4.716149
9,97.837736,919.099393,0.064516,7105.147999,0.870726,9.995367,1.059244,43.733019,0.353064,65.585068,...,Success,NaN,1.456243e+06,17849.9712,122.115865,0.964012,0.029090,129.933354,4.676003,4.721631


In [15]:
df_optim = pd.read_csv("DAC_optimal_simulation_results_testing.csv")
df_optim.columns

Index(['caco3_flow_rate [kmol/hr]', 'calciner_temp [Deg C]',
       'pellet_moisture_frac', 'air_mol_flow_rate [kmol/hr]', 'RH_air',
       'DAC_column_height [m]', 'pellet_diameter [mm]',
       'DAC_ambient_temp [Deg C]', 'e_voidage', 'capture_efficiency_%',
       'SECC_$', 'Simulation_status', 'Failure_reason', 'Heat_duty[cal/sec]',
       'Brake_power[kW]', 'calcium_hydroxide_pellet[kmol_per_hr]',
       'SCM_conversion_fraction', 'Ca(OH)2_conversion',
       'mass_flow_co2_in[kg/hr]', 'mass_flow_co2_out[kg/hr]',
       'DAC_inlet_press[atm]'],
      dtype='object')

In [19]:
df_optim["SECC_$"]

0     10.950542
1     10.950542
2      7.172618
3      5.621132
4      3.874981
5      2.323490
6      7.172618
7      1.046576
8      3.874981
9      5.621132
10     2.123235
11     1.046576
12     2.323490
Name: SECC_$, dtype: float64

In [52]:
df_sim_results = pd.read_csv("DAC_simulation_results.csv")
#df_sim_results[df_sim_results["SCM_conversion_fraction"] > 0].sort_values(by="SCM_conversion_fraction", ascending=False)
df_sim_results.tail(30)

,caco3_flow_rate [kmol/hr],calciner_temp [Deg C],pellet_moisture_frac,air_mol_flow_rate [kmol/hr],RH_air,DAC_column_height [m],pellet_diameter [mm],DAC_ambient_temp [Deg C],e_voidage,sim_index,Simulation_status,Failure_reason,Heat_duty[cal/sec],Brake_power[kW],calcium_hydroxide_pellet[kmol_per_hr],SCM_conversion_fraction,Ca(OH)2_conversion,mass_flow_co2_in[kg/hr],mass_flow_co2_out[kg/hr],DAC_inlet_press[atm]
1070,97.168221,900.856253,0.190224,7788.506024,0.447167,4.215255,4.215585,29.214308,0.388035,1070.0,Success,NaN,1.433222e+06,4344.38391,158.708349,0.028194,0.000857,129.933354,126.270015,1.574627
1071,90.647216,914.015049,0.108440,8812.918594,0.564654,6.392935,1.961374,27.370128,0.431136,1071.0,Success,NaN,1.345820e+06,7980.58817,126.680204,0.127958,0.004168,129.933354,113.307371,2.191824
1072,25.796346,918.775035,0.240289,8949.452809,0.430063,3.097217,4.037014,18.441279,0.444325,1072.0,Success,NaN,3.838981e+05,2421.34293,45.158113,0.013546,0.001550,129.933354,128.173305,1.304308
1073,54.223119,945.203754,0.253156,8448.434588,0.172175,8.484204,2.409842,24.734290,0.386933,1073.0,Success,NaN,8.175278e+05,8930.75903,96.374100,0.063596,0.003463,129.933354,121.670050,2.378031
1074,72.712045,947.436594,0.290168,8589.325887,0.629979,8.265727,1.482146,27.667244,0.360145,1074.0,Success,NaN,1.097489e+06,14740.86630,134.288666,0.275234,0.011175,129.933354,94.171310,3.772536
1075,12.590563,907.422329,0.250577,8466.538188,0.473523,5.937127,3.068056,18.492058,0.431701,1075.0,Success,NaN,1.863180e+05,5355.25669,22.311727,0.027632,0.006479,129.933354,126.343063,1.731726
1076,8.528741,891.472428,0.061513,8054.794168,0.819203,8.640573,2.952510,40.604026,0.441842,1076.0,Failure,"(-2147352567, 'Exception occurred.', (14, 'VAS...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1077,58.022260,896.788198,0.343162,8455.518279,0.584782,2.021474,5.244453,30.298233,0.350535,1077.0,Success,NaN,8.540868e+05,2805.98678,111.794318,0.013851,0.000705,129.933354,128.133651,1.355489
1078,64.592818,919.384739,0.235602,7785.223443,0.567087,1.311530,4.200956,28.046929,0.398108,1078.0,Success,NaN,9.615529e+05,1524.24680,112.421360,0.011226,0.000513,129.933354,128.474779,1.190327
1079,54.019635,930.916755,0.446082,7252.724137,0.862554,7.436926,1.958699,43.586013,0.445454,1079.0,Success,NaN,8.087541e+05,6898.40717,108.899964,0.168953,0.009234,129.933354,107.980766,1.992744
